# Build Measured Intrinsic Wavefront (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-04-28
**Last Modified:** 2026-04-28
**Status:** Draft
**Keywords:** AOS, intrinsic, double-Zernike, FAM, focal plane

## Description

Build an empirical estimate of the focal-plane intrinsic Zernike map
("measured intrinsic") from FAM donut data, by fitting and subtracting
a chosen sparse subset of Double-Zernike modes (focal `k` × pupil `j`).

Key functionality:
1. Load donut + visit-info parquet pair, filter by day_obs / elevation /
   rotator angle / seq_num.
2. Per visit fit the chosen `(k, j)` DZ modes to `data – tabulated_intrinsic`.
3. Subtract the DZ contribution (only); take the median residual on a
   focal-plane grid per pupil j → iter-1 measured intrinsic.
4. Iterate using iter-1 as the new tabulated intrinsic → iter-2.
5. Compare original / iter-1 / iter-2 / tabulated intrinsic per pupil j
   on a common 5–95 % colour scale.
6. Save iter-2 DZ fits and the measured intrinsic map.

**Output:** parquet (or FITS / HDF5 — see § 8) with `(thx, thy)` in OCS
and CCS plus median `zk` per pupil Noll index, plus a parquet with the
iter-2 per-visit DZ fit coefficients.

**Based on:** `code/measured_intrinsic.py`, `code/dz_fitting.py`,
`code/intrinsics_lib.py`, the `run_pipeline` mktable / fit / plots flow.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-28 | Aaron Roodman | Initial version. |
| 2026-05-12 | Aaron Roodman | Major rework: (1) multi-chunk loading driven by `binning` ("bin_2x" default) resolved against `runs.yaml`, with ±3° rotator filter default and any elevation; (2) new OFC SVD / Reachability reference section — V, σ, U, U² matrix plots, reachability heatmap with cell text, per-j summary table, and a reference fit demo on a real visit; (3) **Path A** — `build_measured_intrinsic_uconstrained` fits all (k=1..6) × 21 pupil-j DZ coefficients per visit and projects each visit's coefficient vector onto the first `n_keep` U-modes (`w_proj = U_eff @ U_effᵀ @ w`), iterating `n_iter` times; (4) **Path B** — `build_measured_intrinsic` with `removal_spec` built automatically from `removal_spec_from_reachability` at `reach_threshold` (overridable via `removal_spec_manual`); (5) **Path A per-visit validation** — heatmaps of `W_raw`, `W_fit`, `W_residual` vs visit ordinal; U-mode amplitude heatmap and line plots; per-pupil-j panel plots of `W_fit` and `W_residual` (lines coloured by focal k); (6) **Path A vs Path B comparison** — side-by-side iter-final measured-intrinsic maps and per-pupil-j A−B differences with RMS summary; (7) outputs reorganised under `output/build_measured_intrinsic/{binning}_nkeep_{n_keep}_…/` with every file prefixed `measured_intrinsic_{output_label}_…`. |


## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access — multi-chunk](#data)
5. [Z4 Height Correction (precompute)](#z4height)
6. [OFC Sensitivity SVD & Reachability (reference)](#svd)
    - [6.1 SVD diagnostic plots — V, σ, U, U²](#svd-plots)
    - [6.2 Reachability heatmap & summary table](#reachability)
    - [6.3 Reference fit demo](#fit-demo)
7. [Path B — Reachability-thresholded DZ removal](#analysis-pathB)
8. [Path A — U-mode constrained DZ removal](#analysis-pathA)
9. [Path A — per-visit fit validation](#pathA-validation)
10. [Path A vs Path B — Measured Intrinsic comparison](#compare-AB)
11. [Path B diagnostic plots — 4-panel per pupil j](#plots)
12. [Iter-2 (Path B) Measured Intrinsic — CCS-binned maps](#ccs-maps)
13. [Iter-2 (Path B) Measured Intrinsic — OCS, rotator near 0](#ocs-rot0-maps)
14. [Per-DZ-term Tracking (Iter-1 vs Iter-2, Path B)](#tracking)
15. [Per-visit Residual Validation (Path B)](#validation)
16. [Output Tables](#output)
17. [Output Format Options](#format)


<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# ----- Binning selector -----
#   binning = 'bin_2x'  ->  use the three bin_2x chunks (chunk1, 3, 5)
#   binning = 'bin_1x'  ->  use the three bin_1x chunks (chunk2, 4, 6)
# Explicit `donut_parquets` (below) takes precedence if provided.
binning = 'bin_2x'

# ----- Input donut parquet files (one per pipeline chunk) -----
# If None, the data cell resolves the three chunks for `binning`
# from runs.yaml.  Each donut parquet must have its matching
# `*_visits.parquet` sidecar next to it.
donut_parquets = None         # list[str] or None

# Coordinate system for the fit and grid (FAM triplets are taken near
# rotator angle = 0, so OCS and CCS are essentially equivalent).
coord_sys = 'OCS'

# ----- Filters (None = no constraint) -----
day_obs_min   = None
day_obs_max   = None
alt_min_deg   = None        # elevation in degrees
alt_max_deg   = None
rot_min_deg   = -3.0        # rotator_angle in degrees
rot_max_deg   = +3.0
seq_num_min   = None
seq_num_max   = None

# ----- Per-visit quality cuts (matches run_pipeline fit step) -----
min_donuts        = 500     # require at least this many donuts per visit
bad_fit_threshold = 2.0     # flag bad if any |coeff| > this (μm)

# ============================================================
# OFC sensitivity-matrix / SVD parameters
# ============================================================
# The U-mode constrained path (Path A) and the reachability-thresholded
# path (Path B) both rest on the SVD of the OFC sensitivity matrix S
# (sliced to the k, j of interest) after right-multiplication by the
# OFC normalization matrix.  These parameters set up that SVD.

# Path to the OFC normalization-weights yaml (50-DoF baseline).
# The weights live in ts_config_mttcs (not ts_ofc) at
#   MTAOS/ofc/normalization_weights/range0.5_fwhm-0.15.yaml
# When this parameter is None, the SVD cell auto-discovers it from
# $TS_CONFIG_MTTCS_DIR (the EUPS env var set by
# 'setup ts_config_mttcs').
ofc_normalization_yaml = None

# Focal-Noll k slice (k = 1..6 by default).
k_min, k_max = 1, 6

# Number of U-modes (singular vectors) kept for the projection.
# Equivalent to truncating S to its top n_keep singular components.
n_keep = 50

# Reachability threshold (path B): a (k, j) cell is selected for
# removal if its reachability fraction f_{k,j} >= reach_threshold.
reach_threshold = 0.10

# ----- Removal spec (path B override) -----
# If None, the analysis cell builds the removal spec automatically
# from the reachability grid for the current `reach_threshold`.
# Provide an explicit dict here to override.
removal_spec_manual = None

# ----- Iteration -----
n_iter = 2

# ----- Focal-plane grid -----
n_bins = 73                  # 18*4 + 1 trio-style binning
fp_radius_basis = 1.75       # field radius for Z basis normalization
fp_radius_grid  = 1.8        # grid extent for binned medians

# ----- Z4 CCD-height correction -----
# When `height_map_fits` is set, Z4 is corrected per donut BEFORE the
# DZ fits and the binning:
#   - measured Z4: subtract Z4hgt        = 15 μm/mm × height(fpx, fpy)
#   - tabulated Z4: subtract Z4hgt_transpose
# `Z4hgt_transpose` evaluates the height at the per-CCD (x<->y)
# transpose of (fpx, fpy), to match the bug in the pipeline's
# per-CCD intrinsic Zernike calculation.  Set
# intrinsic_transpose_bug=False once the upstream tabulation is fixed.
height_map_fits          = 'output/LSST_FP_cold_b_measurement_4col_bysurface.fits'
height_to_z4_factor      = None   # None -> ccd_height.HEIGHT_TO_Z4_UM_PER_MM
intrinsic_transpose_bug  = True   # use per-CCD transposed Z4hgt for tabulated

# ----- Per-visit residual validation -----
# Histograms are cheap; the residual color maps are slow because
# `binned_statistic_2d` is called per visit per j.  Default to
# histograms only.
make_residual_maps   = False
residual_j_range     = range(4, 20)     # pupil j = 4..19 (16 panels = 4x4 fully filled)
residual_hist_range  = (-1.0, 1.0)      # μm
residual_n_hist_bins = 60
residual_map_n_bins  = 16               # coarse, like the movie binning
residual_map_fp_radius = 1.8

# Iter-2 measured-intrinsic CCS map pages.  These rebin the
# OCS-frame DZ-subtracted zk values on a (thx_CCS, thy_CCS)
# grid so CCD-fixed features stand out.
ccs_maps_per_page  = 4    # 2x2 layout per page
ccs_map_n_bins     = None # None -> reuse n_bins from §6 (default 73)
ccs_map_fp_radius  = None

# Half-width (deg) for the iter-2 "OCS, rotator near zero"
# map view (§9).  Donuts whose visit has |rotator_angle|
# > rot0_window_deg are dropped before the OCS rebin so
# that view is rotator-smearing-free.
rot0_window_deg    = 3.0 # None -> reuse fp_radius_grid (default 1.8)

# Subsample the per-visit histogram (and map) pages: only
# emit a page every `hist_visit_stride` visits.  Summary
# σ / σ_MAD stats are still computed for every good
# visit and all of those drive the time-series plots.
hist_visit_stride = 12
per_zernike_sigma_j = range(4, 9)        # j = 4..8

# ----- Output -----
output_dir = None            # None -> derive from donut stem + filter tag
output_format = 'parquet'    # parquet (recommended)


<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from astropy.table import QTable, vstack as table_vstack

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

sys.path.insert(0, 'code')
from dz_fitting import derive_noll_indices
from measured_intrinsic import (
    expand_removal_spec,
    apply_visit_filters,
    bin_median_focal,
    interpolate_grid_at_donuts,
    build_measured_intrinsic,
    build_measured_intrinsic_uconstrained,
    removal_spec_from_reachability,
    assemble_intrinsic_table,
    save_dz_fits,
    _fit_one_image_subset,
    _dz_contrib_from_params,
)
from run_pipeline import (
    load_runs as _rp_load_runs,
    load_param_sets as _rp_load_param_sets,
    resolve_run as _rp_resolve_run,
    donut_path as _rp_donut_path,
)
# OFC sensitivity matrix + normalization weights (needed for the SVD
# section that drives the U-mode and reachability paths).  Both are
# only available inside the LSST conda env on RSP; on the laptop we
# fall back to None and skip those cells.
try:
    import yaml
    from lsst.ts.ofc import OFCData
    _ofc_ok = True
except Exception as _e:
    print(f'(OFC / yaml import skipped: {type(_e).__name__}: {_e})')
    _ofc_ok = False

# CCD focal-plane height map helpers (for the Z4 correction).  These
# imports pull in lsst.obs.lsst, sklearn, and astropy.io — they only
# work on RSP / inside the LSST conda env.
try:
    from ccd_height import (
        compute_fp_coords,
        make_metrology_table,
        get_height_interpolator,
        height_to_z4,
        transpose_around_ccd_centers,
        HEIGHT_TO_Z4_UM_PER_MM,
    )
    _ccd_height_ok = True
except Exception as _e:
    print(f'(ccd_height import skipped: {type(_e).__name__}: {_e})')
    _ccd_height_ok = False

print('Loaded measured_intrinsic helpers.')


<a id='functions'></a>
## 3. Helper Functions

In [ ]:
def filter_donuts_by_visits(donut_df, visits_kept):
    """Return rows of donut_df whose (day_obs, seq_num) appears in visits_kept."""
    keep_keys = set(zip(
        np.asarray(visits_kept['day_obs']).tolist(),
        np.asarray(visits_kept['seq_num']).tolist(),
    ))
    keys = list(zip(donut_df['day_obs'].tolist(),
                    donut_df['seq_num'].tolist()))
    mask = np.array([k in keep_keys for k in keys])
    return donut_df.loc[mask].reset_index(drop=True)


def common_color_scale(grids, plo=5.0, phi=95.0):
    """Return (vmin, vmax) covering the plo-phi percentile of *all* grids
    (skip NaNs; ignore empty grids)."""
    pooled = np.concatenate(
        [g.ravel()[~np.isnan(g.ravel())] for g in grids if g is not None
         and np.any(np.isfinite(g))])
    if pooled.size == 0:
        return -1.0, 1.0
    return tuple(np.nanpercentile(pooled, [plo, phi]))


def output_dir_default(binning, n_keep_label, coord_sys,
                       day_obs_min, day_obs_max,
                       seq_num_min, seq_num_max,
                       alt_min_deg, alt_max_deg,
                       rot_min_deg, rot_max_deg):
    """Build a self-describing output subdirectory.

    Top-level: `output/build_measured_intrinsic/`
    Tag: `{binning}_nkeep_{n_keep_label}_{coord_sys}` plus filter
    qualifiers as needed.
    """
    parts = [str(binning), f'nkeep_{n_keep_label}', str(coord_sys)]
    if day_obs_min or day_obs_max:
        parts.append(f'day_{day_obs_min or ""}_{day_obs_max or ""}')
    if seq_num_min or seq_num_max:
        parts.append(f'seq_{seq_num_min or ""}_{seq_num_max or ""}')
    if alt_min_deg is not None or alt_max_deg is not None:
        parts.append(f'alt_{alt_min_deg or ""}_{alt_max_deg or ""}')
    if rot_min_deg is not None or rot_max_deg is not None:
        parts.append(f'rot_{rot_min_deg or ""}_{rot_max_deg or ""}')
    return Path('output') / 'build_measured_intrinsic' / '_'.join(parts)


# ----------------------------------------------------------------------
# Multi-chunk donut parquet resolution
# ----------------------------------------------------------------------

def resolve_chunk_parquets(binning, donut_parquets=None):
    """Return the list of donut parquet paths to use.

    If `donut_parquets` is provided, that wins.  Otherwise we look up
    the runs in `runs.yaml` whose `param_set` matches the requested
    binning (`fam_danish_v1_triplets_bin_2x` or `..._bin_1x`) and
    return the donut paths in chunk order.
    """
    if donut_parquets:
        return [Path(p) for p in donut_parquets]

    target_ps = f'fam_danish_v1_triplets_{binning}'
    runs = _rp_load_runs().get('runs', {})
    param_sets = _rp_load_param_sets()

    paths = []
    for name, cfg in runs.items():
        if cfg.get('param_set') != target_ps:
            continue
        resolved = _rp_resolve_run(cfg, param_sets)
        p = Path(_rp_donut_path(resolved))
        if p.exists():
            paths.append(p)
        else:
            print(f'  (skipping {name}: {p} not found)')
    if not paths:
        raise FileNotFoundError(
            f'No donut parquet files resolved for binning={binning!r}')
    return paths


def visits_sidecar_path(donut_parquet_path):
    """`{stem}_visits.parquet` next to the donut parquet."""
    p = Path(donut_parquet_path)
    return p.with_name(p.stem + '_visits.parquet')


def load_chunks(donut_parquet_paths):
    """Read donut + visits sidecars across chunks, return (donut_df, visits)."""
    donut_frames = []
    visits_tables = []
    for p in donut_parquet_paths:
        vp = visits_sidecar_path(p)
        assert p.exists(), f'missing {p}'
        assert vp.exists(), f'missing {vp}'
        d = pd.read_parquet(p)
        v = QTable.read(str(vp), format='parquet')
        donut_frames.append(d)
        visits_tables.append(v)
        print(f'  {p.name}:  {len(d):>8d} donuts,  {len(v):>4d} visits')
    donut_df = pd.concat(donut_frames, ignore_index=True)
    visits_full = (visits_tables[0]
                   if len(visits_tables) == 1
                   else table_vstack(visits_tables, join_type='exact'))
    return donut_df, visits_full


<a id='data'></a>
## 4. Data Access

In [ ]:
# Resolve the donut parquet list (one per pipeline chunk).
donut_parquet_paths = resolve_chunk_parquets(binning, donut_parquets)
print(f'Using binning = {binning!r};  {len(donut_parquet_paths)} chunk(s):')
for p in donut_parquet_paths:
    print(f'   {p}')

donut_full, visits_full = load_chunks(donut_parquet_paths)
print(f'\nTotal: {len(donut_full)} donuts, {len(visits_full)} visits')

# Apply visit-level filters
visits_kept = apply_visit_filters(
    visits_full,
    day_obs_min=day_obs_min, day_obs_max=day_obs_max,
    alt_min_deg=alt_min_deg, alt_max_deg=alt_max_deg,
    rotator_min_deg=rot_min_deg, rotator_max_deg=rot_max_deg,
    seq_num_min=seq_num_min, seq_num_max=seq_num_max,
)
print(f'Visits after filters: {len(visits_kept)}/{len(visits_full)}')

donut_df = filter_donuts_by_visits(donut_full, visits_kept)
print(f'Donuts after filters: {len(donut_df)}/{len(donut_full)}')

# Derive Noll indices from the data
nZk = np.stack(donut_df[f'zk_{coord_sys}'].values).shape[1]
noll_arr = (np.array(visits_kept['nollIndices'][0])
            if 'nollIndices' in visits_kept.colnames else None)
iZs, iZidx = derive_noll_indices(nZk, noll_arr)
print(f'Pupil Noll indices ({len(iZs)}): {iZs}')

# For the multi-chunk donut frame, use the first chunk's stem as the
# label seed (output_dir_default uses Path(...).stem only).
primary_donut_parquet = str(donut_parquet_paths[0])


<a id='z4height'></a>
## 5. Z4 Height Correction (precompute)

Compute the per-donut Z4 contribution from the focal-plane height map
**before** any DZ fitting or binning:

* `Z4hgt`           = `15 μm/mm × height(fpx, fpy)` — for the measured data.
* `Z4hgt_transpose` = `15 μm/mm × height(fpx_swap, fpy_swap)` where
  `(fpx_swap, fpy_swap)` is the per-CCD x↔y transpose around the
  detector center — for the *tabulated* intrinsic, mirroring the bug
  in the pipeline's per-CCD intrinsic Zernike calculation.

If the metrology FITS file is unavailable or the LSST stack isn't
loaded, the corrections are skipped.  When `intrinsic_transpose_bug` is
False the tabulated correction uses `Z4hgt` directly (no transpose).

In [ ]:
data_offset = None
intrinsic_offset = None
Z4hgt = None
Z4hgt_intrinsic = None

_z4_can_correct = (_ccd_height_ok and height_map_fits
                   and Path(height_map_fits).exists() and 4 in iZs)

if _z4_can_correct:
    print(f'Loading metrology: {height_map_fits}')
    metrology = make_metrology_table(height_map_fits)
    print(f'  {len(metrology)} metrology points across '
          f'{len(set(metrology["det"]))} sensors')

    from lsst.obs.lsst import LsstCam       # local import (LSST stack only)
    camera = LsstCam.getCamera()

    factor = (height_to_z4_factor
              if height_to_z4_factor is not None
              else HEIGHT_TO_Z4_UM_PER_MM)
    interp = get_height_interpolator(metrology)

    # Per-donut focal-plane (mm) coords -> height -> Z4 contribution
    fpx, fpy = compute_fp_coords(donut_df, camera)
    Z4hgt = height_to_z4(interp(fpx, fpy), factor=factor)
    print(f'Z4hgt   per donut (data side): mean={float(np.nanmean(Z4hgt)):.4f} μm  '
          f'std={float(np.nanstd(Z4hgt)):.4f} μm')

    # Intrinsic side: maybe transposed
    if intrinsic_transpose_bug:
        det_names = np.asarray(donut_df['detector']).astype(str)
        fpx_t, fpy_t = transpose_around_ccd_centers(fpx, fpy, det_names, camera)
        Z4hgt_intrinsic = height_to_z4(interp(fpx_t, fpy_t), factor=factor)
        print(f'Z4hgt_t per donut (intrinsic, x<->y per CCD): '
              f'mean={float(np.nanmean(Z4hgt_intrinsic)):.4f} μm  '
              f'std={float(np.nanstd(Z4hgt_intrinsic)):.4f} μm')
    else:
        Z4hgt_intrinsic = Z4hgt
        print('Using un-transposed Z4hgt for the intrinsic '
              '(intrinsic_transpose_bug=False)')

    data_offset = {4: Z4hgt}
    intrinsic_offset = {4: Z4hgt_intrinsic}
    print('Z4 corrections will be applied inside build_measured_intrinsic '
          '(measured Z4 -= Z4hgt; intrinsic Z4 -= '
          f'{"Z4hgt_transpose" if intrinsic_transpose_bug else "Z4hgt"}).')
else:
    if not _ccd_height_ok:
        print('Skipping Z4 height correction: ccd_height module unavailable')
    elif not height_map_fits:
        print('Skipping Z4 height correction: height_map_fits is None')
    elif not Path(height_map_fits).exists():
        print(f'Skipping Z4 height correction: {height_map_fits} not found')
    elif 4 not in iZs:
        print('Skipping Z4 height correction: Z4 not in iZs')

<a id='svd'></a>
## 6. OFC Sensitivity Matrix — SVD & Reachability (reference)

Adapted from `jk_coverage_plots.ipynb`.  We slice the OFC sensitivity
matrix `S_full` of shape `(31, 29, 50)` to the pupil-j indices in
`iZs` and the focal-k slice `k_min..k_max`, then right-multiply by the
diagonal OFC normalization matrix.  The SVD of the resulting matrix
`S` gives:

* **V** — pupil-Zernike-direction basis of the column space; each
  column of V is a unit "v-mode" in DZ space.
* **U** — DoF-direction basis of the row space; each column of U is a
  unit "u-mode" in the 50-DoF actuator/figure space.
* **σ** — singular values relating the two.

`U_eff = U[:, :n_keep]` is the projector onto the first `n_keep`
u-modes.  Right-multiplying by `U_eff @ U_eff.T` projects any DZ
coefficient vector onto the slice of DZ space reachable with `n_keep`
DoF.  The per-(k, j) **reachability**

    f_{k, j} = ||U_eff^T e_{k, j}||²

is the fraction of an isolated unit DZ basis vector that lies in
col(`S`).  This is the quantity thresholded in Path B and used as a
diagnostic in Path A.


In [ ]:
# ============================================================
# Build the OFC sensitivity-matrix SVD for the chosen (k, j) slice
# ============================================================
assert _ofc_ok, ('Need lsst.ts.ofc + yaml — make sure this notebook runs '
                 'inside the LSST conda env on RSP.')

def _find_ofc_normalization_yaml(user_path=None,
                                 name='range0.5_fwhm-0.15.yaml'):
    """Locate the OFC normalization-weights yaml inside ts_config_mttcs.

    Path is $TS_CONFIG_MTTCS_DIR/MTAOS/ofc/normalization_weights/<name>.
    An explicit `user_path` wins if it exists.
    """
    if user_path:
        p = Path(user_path)
        if not p.exists():
            raise FileNotFoundError(
                f'user-supplied ofc_normalization_yaml = {p} does not exist')
        return p

    root = os.environ.get('TS_CONFIG_MTTCS_DIR')
    if not root:
        raise RuntimeError(
            'TS_CONFIG_MTTCS_DIR is not set. Run '
            '`setup -r ~/u/LSST/packages/ts_config_mttcs -j` (or the '
            'equivalent eups setup) before launching the notebook, or '
            'set ofc_normalization_yaml explicitly in the params cell.')
    target = Path(root) / 'MTAOS' / 'ofc' / 'normalization_weights' / name
    if not target.exists():
        raise FileNotFoundError(
            f'OFC normalization yaml not found at {target}. '
            'Try `git pull` in ts_config_mttcs.')
    return target

norm_yaml_path = _find_ofc_normalization_yaml(ofc_normalization_yaml)
print(f'OFC normalization yaml: {norm_yaml_path}')
with open(norm_yaml_path) as _fp:
    normalization_weights = np.array(yaml.safe_load(_fp))
normalization_matrix = np.diag(normalization_weights)
print(f'OFC normalization: {len(normalization_weights)} weights '
      f'(min={normalization_weights.min():.3g}, '
      f'max={normalization_weights.max():.3g})')

ofc_data = OFCData('lsst')
S_full = np.asarray(ofc_data.sensitivity_matrix)        # (n_field=31, n_zk=29, n_dof=50)
print(f'S_full shape (field, pupil-j, DoF) = {S_full.shape}')

# Slice to k = k_min..k_max and pupil-j = iZs.  The OFC convention
# uses Noll k = 1, but its first axis is k-1 -> 0, so [k_min:k_max+1]
# already maps to k = k_min..k_max when k_min >= 1.
#
# Pupil-j axis: index 0 corresponds to Noll j=4 (focus), so we need
# columns iZs - 4.  We use advanced indexing because iZs is generally
# not contiguous (e.g. omits j=20, 21).
iZs_arr = np.asarray(iZs, dtype=int)
n_k = k_max - k_min + 1
n_j = len(iZs_arr)
S_slab = S_full[k_min - 1:k_max, iZs_arr - 4, :]        # (n_k, n_j, n_dof)
n_dof = S_slab.shape[-1]
S = S_slab.reshape(-1, n_dof) @ normalization_matrix
print(f'S (kj-row, dof-col) = {S.shape}   '
      f'[k = {k_min}..{k_max} ({n_k}), pupil-j = {iZs_arr.min()}..{iZs_arr.max()} ({n_j})]')

# Row layout matches jk_coverage_plots:  row = (k - k_min) * n_j + (j_idx).
# Build kj_grid in the same order — used by build_measured_intrinsic_uconstrained.
kj_grid = [(int(k_min + ki), int(iZs_arr[ji]))
           for ki in range(n_k) for ji in range(n_j)]

# SVD
U, Sigma, Vh = np.linalg.svd(S, full_matrices=False)
V = Vh.T
print(f'SVD: U={U.shape}, Σ={Sigma.shape} (max={Sigma[0]:.3g}, min={Sigma[-1]:.3g}), V={V.shape}')

# Effective U at the chosen n_keep
n_keep_eff = min(int(n_keep), U.shape[1])
U_eff = U[:, :n_keep_eff]
print(f'U_eff shape = {U_eff.shape}  (n_keep = {n_keep_eff})')

# Per-(k, j) reachability fraction f_{k, j} = sum_m U_eff[row, m]^2
frac_1d = (U_eff ** 2).sum(axis=1)                      # shape (n_k * n_j,)
frac_2d = frac_1d.reshape(n_k, n_j)                     # (n_k, n_j)

# Convenience labels: rows = k (outer), cols = j (inner)
dz_kj_labels  = [f'k={k_min+ki},j={iZs_arr[ji]}'
                 for ki in range(n_k) for ji in range(n_j)]

# Output label used by every file written by this notebook —
# encodes the binning choice and the U-mode truncation length.
output_label = f'{binning}_nkeep_{n_keep_eff}'
print(f'output_label = {output_label!r}')


<a id='svd-plots'></a>
### 6.1 SVD diagnostic plots — V, σ, U, U²


In [ ]:
# ------------------------------------------------------------
# V matrix (DZ rows × v-mode cols), singular spectrum,
# U matrix (DZ rows × u-mode cols), and U² heatmap.
# ------------------------------------------------------------
from matplotlib.colors import TwoSlopeNorm

def _row_ticks(ax):
    """Major ticks every k boundary; labels '(k_min+i)'."""
    ax.set_yticks([(ki + 0.5) * n_j - 0.5 for ki in range(n_k)],
                  minor=True)
    ax.set_yticks([ki * n_j - 0.5 for ki in range(n_k + 1)])
    ax.set_yticklabels([])
    ax.set_yticks([(ki + 0.5) * n_j - 0.5 for ki in range(n_k)],
                  minor=True)
    ax.set_yticklabels([f'k={k_min+ki}' for ki in range(n_k)], minor=True)
    ax.tick_params(which='minor', length=0)

# --- V matrix --------------------------------------------------------
vmax_V = float(np.max(np.abs(V)))
fig_V, axV = plt.subplots(figsize=(8.5, 6.0), layout='constrained')
im = axV.imshow(V, aspect='auto', cmap='RdBu_r',
                norm=TwoSlopeNorm(vmin=-vmax_V, vcenter=0.0, vmax=vmax_V))
axV.set_title('V — DZ-direction coefficients of each v-mode')
axV.set_xlabel('v-mode index')
axV.set_ylabel('DZ row (k outer, pupil-j inner)')
_row_ticks(axV)
plt.colorbar(im, ax=axV, label='coefficient')

# --- Singular spectrum ----------------------------------------------
fig_sig, axS = plt.subplots(figsize=(7.5, 4.5), layout='constrained')
axS.semilogy(np.arange(1, len(Sigma) + 1), Sigma, marker='o', ms=3)
axS.axvline(n_keep_eff + 0.5, ls='--', color='k', lw=0.8,
            label=f'n_keep = {n_keep_eff}')
axS.set_xlabel('singular-value index')
axS.set_ylabel('σ')
axS.set_title('OFC sensitivity-matrix singular spectrum')
axS.grid(alpha=0.3)
axS.legend()

# --- U matrix --------------------------------------------------------
vmax_U = float(np.max(np.abs(U)))
fig_U, axU = plt.subplots(figsize=(8.5, 6.0), layout='constrained')
im = axU.imshow(U, aspect='auto', cmap='seismic',
                norm=TwoSlopeNorm(vmin=-vmax_U, vcenter=0.0, vmax=vmax_U))
axU.set_title('U — DZ-direction coefficients of each u-mode')
axU.set_xlabel('u-mode index')
axU.set_ylabel('DZ row (k outer, pupil-j inner)')
axU.axvline(n_keep_eff - 0.5, ls='--', color='k', lw=0.8)
_row_ticks(axU)
plt.colorbar(im, ax=axU, label='coefficient')

# --- U² heatmap (energy of each DZ row across u-modes) ---------------
fig_U2, axU2 = plt.subplots(figsize=(8.5, 6.0), layout='constrained')
im = axU2.imshow(U ** 2, aspect='auto', cmap='magma_r',
                 vmin=0.0, vmax=float((U ** 2).max()))
axU2.set_title('U² — energy of each DZ row across u-modes')
axU2.set_xlabel('u-mode index')
axU2.set_ylabel('DZ row (k outer, pupil-j inner)')
axU2.axvline(n_keep_eff - 0.5, ls='--', color='k', lw=0.8)
_row_ticks(axU2)
plt.colorbar(im, ax=axU2, label='U²')

plt.show()


<a id='reachability'></a>
### 6.2 Reachability heatmap & summary table

`f_{k, j}` is the fraction of an isolated DZ basis vector e_{k, j}
that lies in col(S).  Cells with `f >= reach_threshold` are selected
for Path B's removal spec.


In [ ]:
# ------------------------------------------------------------
# Reachability heatmap (cell text = 100 * f_{k, j}) + summary table.
# ------------------------------------------------------------
fig_reach, axR = plt.subplots(
    figsize=(max(8.0, 0.45 * n_j + 2.0), 0.6 * n_k + 2.0),
    layout='constrained')
im = axR.imshow(100.0 * frac_2d, aspect='auto', cmap='magma_r',
                vmin=0.0, vmax=100.0)
axR.set_xticks(range(n_j))
axR.set_xticklabels([f'Z{j}' for j in iZs_arr], rotation=0, fontsize=8)
axR.set_yticks(range(n_k))
axR.set_yticklabels([f'k={k_min+ki}' for ki in range(n_k)])
axR.set_xlabel('pupil Noll j')
axR.set_ylabel('focal Noll k')
axR.set_title(f'Reachability  f_{{k,j}} = ‖U_effᵀ e_{{k,j}}‖²   '
              f'(n_keep = {n_keep_eff})')
for ki in range(n_k):
    for ji in range(n_j):
        val = 100.0 * frac_2d[ki, ji]
        col = 'white' if val > 55 else 'black'
        axR.text(ji, ki, f'{val:4.0f}', ha='center', va='center',
                 fontsize=6, color=col)
plt.colorbar(im, ax=axR, label='Reachability  (%)')

# --- Per-pupil-j summary table (Mean Reachability) -------------------
mean_per_j = frac_2d.mean(axis=0)
n_pass_per_j = (frac_2d >= reach_threshold).sum(axis=0)
print(f'\nReachability summary  '
      f'(n_keep = {n_keep_eff}, threshold = {reach_threshold:g}):\n')
print('   j     mean f       # k passing')
print('   ---   ----------   -----------')
for ji, j in enumerate(iZs_arr):
    print(f'   {int(j):3d}   {mean_per_j[ji]:10.4f}   '
          f'{int(n_pass_per_j[ji]):>11d} / {n_k}')

n_pass_total = int((frac_2d >= reach_threshold).sum())
print(f'\nTotal cells passing threshold: '
      f'{n_pass_total} / {n_k * n_j}')

plt.show()


<a id='fit-demo'></a>
### 6.3 Reference example — fit a full (k = 1..6) × 21 j DZ pattern

To validate the U-mode projection on real data, we draw one measured
visit (the first kept visit) and fit *all* (k, j) cells in `kj_grid`
to its `(zk_data - zk_intrinsic_tab)` per-donut wavefront.  The raw
fit and the U-mode-projected fit are then evaluated on a regular
focal-plane grid and shown in microns, alongside the residual (raw −
projected).


In [ ]:
# ------------------------------------------------------------
# Reference fit demo on the first kept visit
# ------------------------------------------------------------
from measured_intrinsic import (_apply_uconstraint, default_removal_spec)

# Pick a demo visit (first visit in visits_kept)
_demo_dobs = int(visits_kept['day_obs'][0])
_demo_snum = int(visits_kept['seq_num'][0])
_mask_demo = ((donut_df['day_obs'] == _demo_dobs)
              & (donut_df['seq_num'] == _demo_snum)).values
print(f'Demo visit: day_obs={_demo_dobs}, seq_num={_demo_snum}, '
      f'{int(_mask_demo.sum())} donuts')

# Build coefficient parameter dict — all (k, j) in kj_grid
_thx_demo = np.rad2deg(np.asarray(donut_df.loc[_mask_demo, f'thx_{coord_sys}'],
                                  dtype=float))
_thy_demo = np.rad2deg(np.asarray(donut_df.loc[_mask_demo, f'thy_{coord_sys}'],
                                  dtype=float))
_zk_data_demo = np.stack(
    donut_df.loc[_mask_demo, f'zk_{coord_sys}'].values).astype(float)
_zk_intr_demo = np.stack(
    donut_df.loc[_mask_demo, f'zk_intrinsic_{coord_sys}'].values).astype(float)

# Build a spec covering every (k = k_min..k_max, j in iZs_arr) cell.
# expand_removal_spec expects {focal_k: [pupil_j, ...]} so we key it
# by k.  by_pupil (returned) is the inverse layout used downstream.
_full_spec = {int(k_min + ki): [int(j) for j in iZs_arr] for ki in range(n_k)}
_pairs_full, _by_pupil_full, _by_focal_full = expand_removal_spec(_full_spec)

# Fit
_params_raw, _contrib_raw = _fit_one_image_subset(
    _thx_demo, _thy_demo, _zk_data_demo, _zk_intr_demo,
    iZidx, _by_pupil_full, fp_radius_basis,
    _demo_dobs, _demo_snum, 0)

# Project onto first n_keep U-modes
_params_proj, _w_raw, _w_proj = _apply_uconstraint(
    _params_raw, kj_grid, U_eff)
_contrib_proj = _dz_contrib_from_params(
    _params_proj, _by_pupil_full, _thx_demo, _thy_demo,
    iZidx, fp_radius_basis)

# Re-bin both onto the focal grid
_grid_raw, _xb, _yb, _xc, _yc = bin_median_focal(
    _thx_demo, _thy_demo, _contrib_raw, iZidx,
    n_bins=n_bins, fp_radius=fp_radius_grid)
_grid_proj, *_ = bin_median_focal(
    _thx_demo, _thy_demo, _contrib_proj, iZidx,
    n_bins=n_bins, fp_radius=fp_radius_grid)

# Plot raw, projected, residual = raw - projected per pupil j
_nj_show = min(n_j, 6)                  # show first 6 pupil-j for compactness
_js_show = list(iZs_arr[:_nj_show])
fig_demo, axes = plt.subplots(_nj_show, 3,
                              figsize=(11.0, 2.6 * _nj_show),
                              layout='constrained')
for r, j in enumerate(_js_show):
    col_j = iZidx[int(j)]
    g_raw  = _grid_raw[:, :, col_j]
    g_proj = _grid_proj[:, :, col_j]
    g_res  = g_raw - g_proj
    vmax = float(np.nanpercentile(np.abs(g_raw[np.isfinite(g_raw)]), 99))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1e-3
    for c, (g, ttl) in enumerate([
            (g_raw,  f'fit  (j={int(j)})'),
            (g_proj, f'U-proj  (j={int(j)})'),
            (g_res,  f'raw − proj  (j={int(j)})')]):
        ax = axes[r, c]
        im = ax.imshow(g.T, origin='lower', cmap='RdBu_r',
                       vmin=-vmax, vmax=+vmax,
                       extent=[_xb[0], _xb[-1], _yb[0], _yb[-1]])
        ax.set_title(ttl, fontsize=9)
        ax.set_aspect('equal')
        plt.colorbar(im, ax=ax, shrink=0.85, label='μm')
fig_demo.suptitle(f'Reference fit demo  —  visit {_demo_dobs}/{_demo_snum}  '
                  f'(n_keep = {n_keep_eff})', fontsize=11)
plt.show()

# Print before/after coefficient comparison for a few key (k, j)
print('\nSample raw vs. projected coefficients (μm):')
print('   k   j    raw         projected')
for k, j in kj_grid[:8] + kj_grid[-4:]:
    rkey = f'dz_z{j}_c{k}'
    raw = _params_raw.get(rkey, np.nan)
    proj = _params_proj.get(rkey, np.nan)
    print(f'   {k}   {j:>2d}   {raw:+9.4f}   {proj:+9.4f}')


<a id='analysis-pathB'></a>
## 7. Path B — Reachability-thresholded DZ removal


In [ ]:
# ============================================================
# Path B — build_measured_intrinsic with a removal_spec selected
# by reachability threshold.
# ============================================================
if removal_spec_manual is not None:
    removal_spec_B = removal_spec_manual
    print('Path B: using user-supplied removal_spec_manual')
else:
    removal_spec_B = removal_spec_from_reachability(
        frac_2d, range(k_min, k_max + 1), iZs_arr, reach_threshold)
    print(f'Path B: removal_spec built from reachability '
          f'>= {reach_threshold:g}')

_pairs_B, _by_pupil_B, _by_focal_B = expand_removal_spec(removal_spec_B)
print(f'\nDZ removal spec (Path B): {len(_pairs_B)} (k, j) modes')
for k in sorted(_by_focal_B):
    js = _by_focal_B[k]
    print(f'  k={k}: j={js[0]}..{js[-1]} ({len(js)} terms)')

result_B = build_measured_intrinsic(
    donut_df, visits_kept, coord_sys, iZs, removal_spec_B,
    n_iter=n_iter,
    n_bins=n_bins,
    fp_radius_basis=fp_radius_basis,
    fp_radius_grid=fp_radius_grid,
    min_donuts=min_donuts,
    bad_fit_threshold=bad_fit_threshold,
    data_offset=data_offset,
    intrinsic_offset=intrinsic_offset,
)
# Back-compat: keep `result` pointing at Path B so the downstream
# diagnostics / output cells (originally one-path) still work.
result = result_B
print(f'\nPath B iterations completed: {len(result_B["iter_results"])}')
for it_idx, it in enumerate(result_B['iter_results']):
    n_total = len(it['fit_rows'])
    n_bad = sum(1 for r in it['fit_rows'] if r.get('bad_fit', False))
    n_good = n_total - n_bad
    print(f'  iter {it_idx + 1}: {n_good}/{n_total} visits passed '
          f'(min_donuts >= {min_donuts}, |coeff| <= {bad_fit_threshold} μm)')


<a id='analysis-pathA'></a>
## 8. Path A — U-mode constrained DZ removal

Fit *all* (k = `k_min`..`k_max`) × `len(iZs)` pupil-j DZ coefficients
on each visit, project the coefficient vector onto the first
`n_keep` U-modes (`w_proj = U_eff @ U_effᵀ @ w`), then reconstruct
the per-donut DZ contribution from `w_proj` and subtract it from the
measured Zernike maps.  Iterates `n_iter` times.


In [ ]:
# ============================================================
# Path A — U-mode-constrained build of the measured intrinsic.
# ============================================================
result_A = build_measured_intrinsic_uconstrained(
    donut_df, visits_kept, coord_sys, iZs,
    kj_grid=kj_grid,
    U_eff=U_eff,
    n_iter=n_iter,
    n_bins=n_bins,
    fp_radius_basis=fp_radius_basis,
    fp_radius_grid=fp_radius_grid,
    min_donuts=min_donuts,
    bad_fit_threshold=bad_fit_threshold,
    data_offset=data_offset,
    intrinsic_offset=intrinsic_offset,
)
print(f'\nPath A iterations completed: {len(result_A["iter_results"])}')
for it_idx, it in enumerate(result_A['iter_results']):
    n_total = len(it['fit_rows'])
    n_bad   = sum(1 for r in it['fit_rows'] if r.get('bad_fit', False))
    n_good  = n_total - n_bad
    print(f'  iter {it_idx + 1}: {n_good}/{n_total} visits passed '
          f'(min_donuts >= {min_donuts}, |coeff| <= {bad_fit_threshold} μm)')


<a id='pathA-validation'></a>
## 9. Path A — per-visit fit validation

Three classes of diagnostics for the U-mode-constrained per-visit
fit.  For each iteration `it`, stack the per-visit fit results into
`(n_visits, n_kj)` arrays:

* `W_raw[v, m]`  — raw least-squares coefficient for `(k, j) = kj_grid[m]`
* `W_fit[v, m]`  — the projected coefficient that is actually subtracted
  (`= U_eff @ U_effᵀ @ W_raw[v]` for each visit `v`)
* `W_residual = W_raw − W_fit`  — the part of the fit that the
  U-mode truncation discards

Plus the per-visit U-mode amplitudes `A_modes = W_raw @ U_eff` of
shape `(n_visits, n_keep_eff)`.  Plots:

1. Heatmaps of `W_raw`, `W_fit`, `W_residual` (rows = `(k, j)`,
   cols = visit ordinal).
2. Heatmap of `A_modes` + line plots of the first 12 U-mode
   amplitudes vs visit ordinal.
3. Per-pupil-j panel plots of `W_fit` and `W_residual` (one panel per
   `j`, lines coloured by focal `k`).


In [ ]:
# ============================================================
# Path A — per-visit fit validation diagnostics
# ============================================================
from matplotlib.cm import get_cmap


def _stack_per_visit_coeffs(fit_rows_raw, fit_rows_proj, kj_grid):
    """Pull (n_visits, n_kj) arrays out of the per-visit param dicts."""
    n_v = len(fit_rows_raw)
    n_m = len(kj_grid)
    W_raw  = np.full((n_v, n_m), np.nan)
    W_fit  = np.full((n_v, n_m), np.nan)
    for v, (rr, rp) in enumerate(zip(fit_rows_raw, fit_rows_proj)):
        for m, (k, j) in enumerate(kj_grid):
            key = f'dz_z{j}_c{k}'
            W_raw[v, m] = float(rr.get(key, np.nan))
            W_fit[v, m] = float(rp.get(key, np.nan))
    return W_raw, W_fit


def _heatmap_visit_vs_kj(W, title, n_k, n_j):
    n_v = W.shape[0]
    vmax = float(np.nanpercentile(np.abs(W), 98)) if np.any(np.isfinite(W)) else 1e-3
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1e-3
    fig, ax = plt.subplots(figsize=(11.5, 6.5), layout='constrained')
    im = ax.imshow(W.T, aspect='auto', cmap='RdBu_r',
                   vmin=-vmax, vmax=+vmax,
                   extent=[-0.5, n_v - 0.5, n_k * n_j - 0.5, -0.5])
    ax.set_xlabel('Visit ordinal')
    ax.set_ylabel('(k, j) row  —  k outer, pupil-j inner')
    ax.set_title(title)
    for ki in range(1, n_k):
        ax.axhline(ki * n_j - 0.5, color='black', lw=0.5, alpha=0.5)
    # k-labels in the row band
    for ki in range(n_k):
        ax.text(-0.01 * n_v, (ki + 0.5) * n_j - 0.5,
                f'k={k_min+ki}', ha='right', va='center',
                fontsize=8, transform=ax.transData)
    plt.colorbar(im, ax=ax, label='μm')
    return fig


def _modes_heatmap(A_modes, title):
    n_v, n_keep_ = A_modes.shape
    vmax = float(np.nanpercentile(np.abs(A_modes), 98))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1e-3
    fig, ax = plt.subplots(figsize=(11.5, 5.0), layout='constrained')
    im = ax.imshow(A_modes.T, aspect='auto', cmap='RdBu_r',
                   vmin=-vmax, vmax=+vmax,
                   extent=[-0.5, n_v - 0.5, n_keep_ - 0.5, -0.5])
    ax.set_xlabel('Visit ordinal')
    ax.set_ylabel('U-mode index')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='Uᵀ w_raw  (μm units)')
    return fig


def _modes_lineplots(A_modes, title, n_show=12):
    n_v, n_keep_ = A_modes.shape
    n_show = min(n_show, n_keep_)
    rows_p = (n_show + 2) // 3
    fig, axes = plt.subplots(rows_p, 3, figsize=(11.5, 1.9 * rows_p),
                             layout='constrained', sharex=True)
    axes = np.atleast_2d(axes).reshape(rows_p, 3)
    cmap = get_cmap('tab20')
    for mi in range(rows_p * 3):
        r, c = divmod(mi, 3)
        ax = axes[r, c]
        if mi >= n_show:
            ax.axis('off')
            continue
        ax.plot(np.arange(n_v), A_modes[:, mi], '.-',
                ms=2, lw=0.7, color=cmap(mi % 20))
        ax.axhline(0, color='k', lw=0.4, alpha=0.5)
        ax.set_title(f'U-mode {mi}', fontsize=9)
        ax.grid(alpha=0.3)
    for ax in axes[-1]:
        ax.set_xlabel('Visit ordinal')
    fig.suptitle(title, fontsize=11)
    return fig


def _per_j_panels(W, title, n_k, n_j, iZs_arr, k_min):
    n_v = W.shape[0]
    cols = 5
    rows_p = (n_j + cols - 1) // cols
    fig, axes = plt.subplots(rows_p, cols,
                             figsize=(14.0, 2.3 * rows_p),
                             layout='constrained', sharex=True)
    axes = np.atleast_2d(axes).reshape(rows_p, cols)
    cmap_k = get_cmap('viridis')
    for ji, j in enumerate(iZs_arr):
        r, c = divmod(ji, cols)
        ax = axes[r, c]
        for ki in range(n_k):
            m = ki * n_j + ji
            ax.plot(np.arange(n_v), W[:, m], '.-', ms=1.5, lw=0.6,
                    color=cmap_k(ki / max(1, n_k - 1)),
                    label=(f'k={k_min+ki}' if ji == 0 else None))
        ax.set_title(f'pupil j={int(j)}', fontsize=8)
        ax.axhline(0, color='k', lw=0.4, alpha=0.5)
        ax.grid(alpha=0.3)
    # Hide any extra panels
    for ji in range(n_j, rows_p * cols):
        axes[ji // cols, ji % cols].axis('off')
    # Single legend pulled from the (0, 0) panel
    axes[0, 0].legend(fontsize=6, loc='upper right', framealpha=0.85)
    for ax in axes[-1]:
        ax.set_xlabel('Visit ordinal')
    fig.suptitle(title, fontsize=11)
    return fig


# ----- Build all validation figures for Path A (per iteration) -----
validation_A_figs = []

for it_idx, it in enumerate(result_A['iter_results']):
    W_raw, W_fit = _stack_per_visit_coeffs(
        it['fit_rows_raw'], it['fit_rows'], kj_grid)
    W_resid = W_raw - W_fit

    # U-mode amplitudes from the raw fit (NaNs treated as zero for the
    # projection, since the U-projection inside the builder did the same).
    A_modes = (np.where(np.isfinite(W_raw), W_raw, 0.0) @ U_eff)

    n_v = W_raw.shape[0]
    print(f'iter {it_idx + 1}: {n_v} visits, n_kj = {len(kj_grid)}, '
          f'n_keep_eff = {n_keep_eff}')

    tag = f'(Path A, iter {it_idx + 1})'
    validation_A_figs.append(
        _heatmap_visit_vs_kj(W_raw,   f'w_raw — fitted DZ coefficients  {tag}',   n_k, n_j))
    validation_A_figs.append(
        _heatmap_visit_vs_kj(W_fit,   f'w_fit — U-projected DZ coefficients  {tag}', n_k, n_j))
    validation_A_figs.append(
        _heatmap_visit_vs_kj(W_resid, f'w_residual = raw − U-projected  {tag}',   n_k, n_j))

    validation_A_figs.append(
        _modes_heatmap(A_modes,
                       f'U-mode amplitudes  A = Uᵀ w_raw  {tag}'))
    validation_A_figs.append(
        _modes_lineplots(A_modes,
                         f'First 12 U-mode amplitudes vs visit ordinal  {tag}'))

    validation_A_figs.append(
        _per_j_panels(W_fit,   f'w_fit per pupil j vs visit ordinal  {tag}',
                      n_k, n_j, iZs_arr, k_min))
    validation_A_figs.append(
        _per_j_panels(W_resid, f'w_residual per pupil j vs visit ordinal  {tag}',
                      n_k, n_j, iZs_arr, k_min))

print(f'\nBuilt {len(validation_A_figs)} Path A validation figures '
      f'({len(validation_A_figs) // n_iter} per iteration × {n_iter} iters).')
plt.show()


<a id='compare-AB'></a>
## 9. Path A vs Path B — Measured Intrinsic comparison


In [ ]:
# ============================================================
# Per-pupil-j comparison of the iter-(n_iter) measured intrinsic grids
# between Path A (U-mode constrained) and Path B (reachability-thresholded).
# ============================================================
_grid_A = result_A['iter_results'][-1]['measured_grid']
_grid_B = result_B['iter_results'][-1]['measured_grid']
_xb_, _yb_ = result_A['xbins'], result_A['ybins']

# Compute a common color scale per panel column.  Δ uses a tighter
# percentile because it's typically small.
def _vmax_pct(arrs, p=99.0):
    pooled = np.concatenate(
        [a.ravel()[np.isfinite(a.ravel())] for a in arrs if a is not None])
    return float(np.nanpercentile(np.abs(pooled), p)) if pooled.size else 1e-3

_js_show = [int(j) for j in iZs]
_cols = 4
_rows = (len(_js_show) + _cols - 1) // _cols

# --- Page 1: Path A iter-final maps ---------------------------------
fig_A, axA = plt.subplots(_rows, _cols,
                          figsize=(3.0 * _cols, 2.6 * _rows),
                          layout='constrained')
axA = np.asarray(axA).reshape(_rows, _cols)
vmaxA = _vmax_pct([_grid_A[:, :, iZidx[j]] for j in _js_show], p=98)
for idx, j in enumerate(_js_show):
    r, c = divmod(idx, _cols)
    ax = axA[r, c]
    im = ax.imshow(_grid_A[:, :, iZidx[j]].T, origin='lower', cmap='RdBu_r',
                   vmin=-vmaxA, vmax=+vmaxA,
                   extent=[_xb_[0], _xb_[-1], _yb_[0], _yb_[-1]])
    ax.set_title(f'A  Z{j}', fontsize=9)
    ax.set_aspect('equal')
for idx in range(len(_js_show), _rows * _cols):
    axA[idx // _cols, idx % _cols].axis('off')
fig_A.colorbar(im, ax=axA.ravel().tolist(), shrink=0.6, label='μm')
fig_A.suptitle(f'Path A (U-mode constrained, n_keep={n_keep_eff}) '
               f'— iter {n_iter} measured intrinsic', fontsize=11)

# --- Page 2: Path B iter-final maps ---------------------------------
fig_B, axB = plt.subplots(_rows, _cols,
                          figsize=(3.0 * _cols, 2.6 * _rows),
                          layout='constrained')
axB = np.asarray(axB).reshape(_rows, _cols)
vmaxB = _vmax_pct([_grid_B[:, :, iZidx[j]] for j in _js_show], p=98)
for idx, j in enumerate(_js_show):
    r, c = divmod(idx, _cols)
    ax = axB[r, c]
    im = ax.imshow(_grid_B[:, :, iZidx[j]].T, origin='lower', cmap='RdBu_r',
                   vmin=-vmaxB, vmax=+vmaxB,
                   extent=[_xb_[0], _xb_[-1], _yb_[0], _yb_[-1]])
    ax.set_title(f'B  Z{j}', fontsize=9)
    ax.set_aspect('equal')
for idx in range(len(_js_show), _rows * _cols):
    axB[idx // _cols, idx % _cols].axis('off')
fig_B.colorbar(im, ax=axB.ravel().tolist(), shrink=0.6, label='μm')
fig_B.suptitle(f'Path B (reachability >= {reach_threshold:g}) '
               f'— iter {n_iter} measured intrinsic', fontsize=11)

# --- Page 3: A − B differences --------------------------------------
fig_D, axD = plt.subplots(_rows, _cols,
                          figsize=(3.0 * _cols, 2.6 * _rows),
                          layout='constrained')
axD = np.asarray(axD).reshape(_rows, _cols)
_diffs = [_grid_A[:, :, iZidx[j]] - _grid_B[:, :, iZidx[j]] for j in _js_show]
vmaxD = _vmax_pct(_diffs, p=99)
for idx, j in enumerate(_js_show):
    r, c = divmod(idx, _cols)
    ax = axD[r, c]
    im = ax.imshow(_diffs[idx].T, origin='lower', cmap='RdBu_r',
                   vmin=-vmaxD, vmax=+vmaxD,
                   extent=[_xb_[0], _xb_[-1], _yb_[0], _yb_[-1]])
    rms = float(np.nanstd(_diffs[idx]))
    ax.set_title(f'A−B  Z{j}\nrms = {rms:.3f} μm', fontsize=8)
    ax.set_aspect('equal')
for idx in range(len(_js_show), _rows * _cols):
    axD[idx // _cols, idx % _cols].axis('off')
fig_D.colorbar(im, ax=axD.ravel().tolist(), shrink=0.6, label='μm')
fig_D.suptitle('Difference  Path A − Path B  (iter-final measured intrinsic)',
               fontsize=11)

# Per-j rms summary table
print('\nRMS of (A − B) per pupil j  (μm):')
print('   j     rms')
print('   ---   --------')
for j, d in zip(_js_show, _diffs):
    print(f'   {j:3d}   {float(np.nanstd(d)):8.4f}')

plt.show()

# Stash for use in the output cell
compare_figs = [fig_A, fig_B, fig_D]


<a id='plots'></a>
## 7. Comparison Plots — 4-panel per pupil j (2x2)

For each pupil Noll j (in `iZs`):

| Panel 1 | Panel 2 | Panel 3 | Panel 4 |
|---|---|---|---|
| Original median (raw `zk`) | Iter-1 measured | Iter-2 measured | Tabulated intrinsic |

Panel 1 uses its own 5–95 % colour scale; panels 2–4 share a single
5–95 % range pooled across those three.  When the Z4 correction is on,
the j=4 page shows `Z4 − Z4hgt` and `Z4_intrinsic − Z4hgt_transpose`.

In [ ]:
def plot_4panel_for_j(j, original_grid, iter1_grid, iter2_grid,
                       tabulated_grid, xbins, ybins,
                       coord_sys, plo=5.0, phi=95.0):
    """4-panel comparison page for one pupil j on a 2x2 grid.

    Color scales:
      * Panel 1 (Original median) uses its own 5-95 % range — the raw
        zk often has a much larger dynamic range than the intrinsic.
      * Panels 2-4 (Iter-1, Iter-2, Tabulated) share a single 5-95 %
        range pooled over those three panels so they are directly
        comparable.
    """
    panels = [
        ('Original median (raw zk)', original_grid),
        ('Iter-1 measured', iter1_grid),
        ('Iter-2 measured', iter2_grid),
        ('Tabulated intrinsic', tabulated_grid),
    ]
    # Panel 1: own range
    if original_grid is not None and np.any(np.isfinite(original_grid)):
        vmin0, vmax0 = np.nanpercentile(original_grid, [plo, phi])
    else:
        vmin0, vmax0 = -1.0, 1.0
    # Panels 2-4: shared range
    intrinsic_grids = [iter1_grid, iter2_grid, tabulated_grid]
    vmin_i, vmax_i = common_color_scale(intrinsic_grids, plo=plo, phi=phi)

    fig, axes = plt.subplots(2, 2, figsize=(13, 11),
                             layout='constrained')
    axes = axes.ravel()
    for idx, (ax, (title, grid)) in enumerate(zip(axes, panels)):
        if grid is None or not np.any(np.isfinite(grid)):
            ax.set_visible(False)
            continue
        if idx == 0:
            vmin, vmax = vmin0, vmax0
            scale_note = f'5-95% own = [{vmin0:.3f}, {vmax0:.3f}] μm'
        else:
            vmin, vmax = vmin_i, vmax_i
            scale_note = f'shared 5-95% = [{vmin_i:.3f}, {vmax_i:.3f}] μm'
        im = ax.imshow(grid.T, origin='lower',
                       extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
                       cmap='viridis', interpolation='none', aspect='equal',
                       vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, shrink=0.8, label='μm')
        ax.set_xlabel(f'thy_{coord_sys} [deg]')
        ax.set_ylabel(f'thx_{coord_sys} [deg]')
        ax.set_title(f'{title}\n({scale_note})', fontsize=10)
    fig.suptitle(
        f'Pupil Z{j}  ({coord_sys})  — Original uses own 5-95%; '
        f'Iter-1 / Iter-2 / Tabulated share a single 5-95%',
        fontsize=12)
    return fig


# Build & display the comparison figures
comparison_figs = []
xbins, ybins = result['xbins'], result['ybins']
iter1_grids = result['iter_results'][0]['measured_grid']
iter2_grids = (result['iter_results'][1]['measured_grid']
               if len(result['iter_results']) >= 2
               else result['iter_results'][-1]['measured_grid'])

for j in iZs:
    if j not in by_pupil:
        # not in removal spec; skip (no DZ subtraction defined)
        continue
    fig = plot_4panel_for_j(
        j,
        result['original_median'].get(j),
        iter1_grids.get(j),
        iter2_grids.get(j),
        result['tabulated_median'].get(j),
        xbins, ybins, coord_sys)
    comparison_figs.append(fig)
    plt.show()

print(f'Built {len(comparison_figs)} per-pupil-j comparison figures')

<a id='ccs-maps'></a>
## 8. Iter-2 Measured Intrinsic — CCS-binned maps

Same iter-2 measured-intrinsic data as §7 (per-donut zk in the OCS
basis after DZ subtraction), but **binned in CCS field-angle
coordinates** so CCD-fixed features stand out.

For FAM triplets taken at rotator angle ≈ 0 the CCS and OCS frames
are nearly identical, so this view differs from §7 mainly to the
extent that any rotator spread is present.  Pages have a 2×2 grid
(four pupil j per page); colour scale per panel is the local
5-95 percentile.  All `iZs` (the `nollIndices` from the fits parquet)
are covered.  Streamed directly to its own PDF.

In [ ]:
# Names for panel titles (local copy — keeps these sections self-contained).
from common.zernike_names import (
    NOLL_NAMES, NOLL_FORMULAS, FOCAL_NAMES, PUPIL_NAMES,
)
def _resolve_output_dir():
    """Resolve and create output_dir if it has not been set up yet.

    Mirrors the snippet in §11 / validation-cell so any of the
    intermediate output sections can run on its own.
    """
    global output_dir
    if output_dir is None:
        output_dir = output_dir_default(
            binning, n_keep_eff, coord_sys,
            day_obs_min, day_obs_max,
            seq_num_min, seq_num_max,
            alt_min_deg, alt_max_deg,
            rot_min_deg, rot_max_deg)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


def render_intrinsic_panel_pdf(grid_dict, xbins, ybins, iZs,
                                output_pdf_path, frame_label,
                                page_subtitle,
                                panels_per_page=4, ncols=2,
                                show_first=True):
    """Stream a multi-page PDF of pupil-j panels for an iter-2 grid.

    Each page has up to `panels_per_page` panels in a `ncols`-wide grid;
    each panel uses its own 5-95 percentile color scale.  Pages are
    written one at a time and the figure is closed after savefig so
    memory stays bounded.

    Parameters
    ----------
    grid_dict : dict {pupil_j: 2D array}
    xbins, ybins : 1D arrays of bin edges
    iZs : list of pupil_j
    output_pdf_path : str / Path
    frame_label : 'OCS' or 'CCS' — used in axis labels
    page_subtitle : str — used in the per-page suptitle
    """
    n_total_pages = int(np.ceil(len(iZs) / max(1, int(panels_per_page))))
    n_pages = 0
    Path(output_pdf_path).parent.mkdir(parents=True, exist_ok=True)
    with PdfPages(output_pdf_path) as pdf:
        for start in range(0, len(iZs), panels_per_page):
            page_js = iZs[start:start + panels_per_page]
            nrows = (len(page_js) + ncols - 1) // ncols
            fig, axes = plt.subplots(
                nrows, ncols,
                figsize=(7.0 * ncols, 6.0 * nrows),
                layout='constrained')
            axes = np.atleast_1d(axes).ravel()
            for idx, j in enumerate(page_js):
                ax = axes[idx]
                grid = grid_dict.get(j)
                if grid is None or not np.any(np.isfinite(grid)):
                    ax.set_visible(False)
                    continue
                vmin, vmax = np.nanpercentile(grid, [5, 95])
                im = ax.imshow(
                    grid.T, origin='lower',
                    extent=[xbins[0], xbins[-1],
                            ybins[0], ybins[-1]],
                    cmap='viridis', interpolation='none', aspect='equal',
                    vmin=vmin, vmax=vmax)
                plt.colorbar(im, ax=ax, shrink=0.8, label='μm')
                ax.set_xlabel(f'thy_{frame_label} [deg]')
                ax.set_ylabel(f'thx_{frame_label} [deg]')
                pname = PUPIL_NAMES.get(j, f'Z{j}')
                ax.set_title(
                    f'Z{j} ({pname})  —  iter-2 measured intrinsic\n'
                    f'5-95% = [{vmin:+.3f}, {vmax:+.3f}] μm',
                    fontsize=11)
            for idx in range(len(page_js), len(axes)):
                axes[idx].set_visible(False)
            page_idx = start // panels_per_page + 1
            fig.suptitle(
                f'{page_subtitle}  ({page_idx}/{n_total_pages})',
                fontsize=12)
            if show_first and n_pages == 0:
                plt.show()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            n_pages += 1
    print(f'Wrote {n_pages} pages -> {output_pdf_path}')
    return n_pages


def _build_ccs_grid(donut_df, result, iZidx, n_bins, fp_radius):
    """Bin the iter-2 wfd_subtracted on a (thx_CCS, thy_CCS) grid.

    Excludes donuts from bad-flagged visits (uses the
    `good_donut_mask` saved by build_measured_intrinsic).
    """
    if 'thx_CCS' not in donut_df.columns or 'thy_CCS' not in donut_df.columns:
        raise KeyError(
            "donut parquet has no thx_CCS / thy_CCS columns — cannot make "
            "the CCS-binned map.  Re-run mktable to add them.")
    final_iter = result['iter_results'][-1]
    wfd_sub = final_iter['wfd_subtracted']
    gd_mask = final_iter.get(
        'good_donut_mask',
        np.ones(len(donut_df), dtype=bool))
    thx = np.rad2deg(np.asarray(donut_df['thx_CCS'], dtype=float))
    thy = np.rad2deg(np.asarray(donut_df['thy_CCS'], dtype=float))
    return bin_median_focal(
        thx[gd_mask], thy[gd_mask],
        wfd_sub[gd_mask], iZidx,
        n_bins=n_bins, fp_radius=fp_radius)


# Reuse §6 grid params unless overridden in the parameters cell.
_ccs_n_bins     = ccs_map_n_bins    or n_bins
_ccs_fp_radius  = ccs_map_fp_radius or fp_radius_grid

print('Building CCS-binned iter-2 measured-intrinsic grids...')
ccs_grid, ccs_xbins, ccs_ybins, ccs_xcent, ccs_ycent = _build_ccs_grid(
    donut_df, result, result['iZidx'],
    _ccs_n_bins, _ccs_fp_radius)
print(f'  binned {result["iZs"]!r} on a {_ccs_n_bins}x{_ccs_n_bins} CCS grid')

_resolve_output_dir()
output_pdf_ccs_maps = str(
    output_dir / 'measured_intrinsic_ccs_intrinsic_maps.pdf')

render_intrinsic_panel_pdf(
    ccs_grid, ccs_xbins, ccs_ybins, iZs,
    output_pdf_ccs_maps,
    frame_label='CCS',
    page_subtitle=('Iter-2 measured intrinsic, binned in CCS  —  '
                   'CCD-fixed features should stand out'),
    panels_per_page=ccs_maps_per_page, ncols=2,
    show_first=True)

<a id='ocs-rot0-maps'></a>
## 9. Iter-2 Measured Intrinsic — OCS, rotator near 0

Same iter-2 per-donut data (`wfd_subtracted` from §6, in the OCS basis)
binned on the OCS field-angle grid, but **restricted to donuts whose
visit has `|rotator_angle| ≤ rot0_window_deg`** (default ±3°).
Removing the rotator-spread donuts gives an OCS view that does not
suffer from rotator smearing — useful as a comparison against the
all-rotator OCS iter-2 grid in §6 / §7.

Pages have a 2×2 grid (four pupil j per page) with per-panel 5–95 %
color scaling.  Streamed directly to its own PDF
(`measured_intrinsic_ocs_rot0_intrinsic_maps.pdf`).

In [ ]:
# Build a rotator-window mask: visits whose |rotator_angle| <= rot0_window_deg.
if 'rotator_angle' not in visits_kept.colnames:
    raise KeyError(
        "visits_kept has no rotator_angle column — cannot build the "
        "OCS rot~0 view.")
vk_dobs = np.asarray(visits_kept['day_obs']).astype(int)
vk_snum = np.asarray(visits_kept['seq_num']).astype(int)
vk_rot  = np.asarray(visits_kept['rotator_angle'], dtype=float)

vk_keep = np.abs(vk_rot) <= rot0_window_deg
rot0_keys = set(zip(vk_dobs[vk_keep].tolist(),
                     vk_snum[vk_keep].tolist()))
print(f'Visits with |rotator_angle| <= {rot0_window_deg}°: '
      f'{int(vk_keep.sum())}/{len(visits_kept)}')

fd_dobs = np.asarray(donut_df['day_obs']).astype(int)
fd_snum = np.asarray(donut_df['seq_num']).astype(int)
rot0_donut_mask = np.array([
    (int(d), int(s)) in rot0_keys
    for d, s in zip(fd_dobs, fd_snum)])

# Combine with the bad-fit good_donut_mask from iter-2 so we use exactly
# the same set of "good" donuts as the OCS iter-2 grid in §6, just
# further restricted to the rotator window.
final_iter = result['iter_results'][-1]
gd_mask = final_iter.get(
    'good_donut_mask', np.ones(len(donut_df), dtype=bool))
combined_mask = rot0_donut_mask & gd_mask
print(f'Donuts kept (rotator window AND not bad-flagged): '
      f'{int(combined_mask.sum())}/{len(donut_df)}')

# Bin in OCS using exactly the §6 grid sampling.
thx = np.rad2deg(np.asarray(donut_df['thx_OCS'], dtype=float))
thy = np.rad2deg(np.asarray(donut_df['thy_OCS'], dtype=float))
wfd_sub = final_iter['wfd_subtracted']

ocs_rot0_grid, ocs_xbins, ocs_ybins, ocs_xcent, ocs_ycent = bin_median_focal(
    thx[combined_mask], thy[combined_mask],
    wfd_sub[combined_mask], result['iZidx'],
    n_bins=n_bins, fp_radius=fp_radius_grid)

_resolve_output_dir()
output_pdf_ocs_rot0 = str(
    output_dir / 'measured_intrinsic_ocs_rot0_intrinsic_maps.pdf')

render_intrinsic_panel_pdf(
    ocs_rot0_grid, ocs_xbins, ocs_ybins, iZs,
    output_pdf_ocs_rot0,
    frame_label='OCS',
    page_subtitle=(f'Iter-2 measured intrinsic, OCS  —  '
                   f'rotator within ±{rot0_window_deg:g}°'),
    panels_per_page=ccs_maps_per_page, ncols=2,
    show_first=False)

<a id='tracking'></a>
## 10. Per-DZ-term Tracking — Iter-1 vs Iter-2

Two PDFs, both 4×4 grids per page over the (k, j) modes in the removal
spec:

* **`measured_intrinsic_tracking.pdf`** — fit coefficient vs visit, with
  thin error bars from the fit `_err` column.  Visits flagged as bad
  (`min_donuts < 500` or any `|coeff| > 2 μm`) are excluded so the plot
  shows exactly what contributed to the median maps.
* **`measured_intrinsic_tracking_robust_rms.pdf`** — per-coefficient
  robust RMS vs visit.  The robust RMS is the standard error from the
  Huber RLM fit (`dz_z{j}_c{k}_err`) — a MAD-equivalent dispersion
  estimator.

Iter-1 = filled blue circles, iter-2 = open red squares.

In [ ]:
def _good_indices(fit_rows):
    """Indices of fit_rows that passed the quality cut (bad_fit == False)."""
    return [i for i, r in enumerate(fit_rows) if not r.get('bad_fit', False)]


def _series(rows, idx, k, j, dz_prefix='dz', err=False):
    suffix = '_err' if err else ''
    col = f'{dz_prefix}_z{j}_c{k}{suffix}'
    return np.array([float(rows[i].get(col, np.nan)) for i in idx])


def _add_day_obs_dividers(ax, day_obs_seq, label_top=False):
    """Vertical dotted lines + tiny rotated day_obs labels at day boundaries."""
    if len(day_obs_seq) < 2:
        return
    if label_top:
        ax.figure.canvas.draw_idle()
    ymax = ax.get_ylim()[1]
    for i in range(1, len(day_obs_seq)):
        if day_obs_seq[i] != day_obs_seq[i - 1]:
            ax.axvline(i - 0.5, color='black', linewidth=0.5,
                       linestyle=':', alpha=0.45, zorder=0)
            if label_top:
                ax.text(i - 0.5, ymax, str(day_obs_seq[i]),
                        fontsize=5, rotation=90, va='top', ha='right',
                        alpha=0.7, zorder=1)


# ---- Shared visit-marker scheme (defined in intrinsics_lib) ----
#
#   color    = elevation bucket (30, 40, 50, 60, 70 deg ±5°)
#   shape    = chunky arrow pointing at the rotator angle
#              (-60..+60 deg in 9 buckets)
#   edge     = filter band (i = no distinct edge; others = colored edge)
#   fill     = iter-1 (filled) vs iter-2 (open)
#
# These were originally defined inline here; they now live in
# intrinsics_lib so all per-visit AOS plots share the same conventions.
from intrinsics_lib import (
    classify_visit, visit_marker_style,
    build_visit_marker_lookup, markers_legend_figure,
)


def _per_visit_groups(fit_rows, idx_subset, visit_meta):
    """Parallel (day_obs, elev, rot, band) lists for an index subset."""
    dobs, elev, rot, band = [], [], [], []
    for i in idx_subset:
        r = fit_rows[i]
        d = int(r['day_obs']); s = int(r['seq_num'])
        m = visit_meta.get((d, s),
                           {'elev': None, 'rot': None, 'band': None})
        dobs.append(d)
        elev.append(m['elev'])
        rot.append(m['rot'])
        band.append(m['band'])
    return dobs, elev, rot, band


def _make_sigma_time_plot(visits_x, yvals, dobs_seq,
                          elev_groups, rot_groups, band_groups,
                          ylabel, title):
    fig, ax = plt.subplots(figsize=(14, 5), layout='constrained')
    for i in range(len(visits_x)):
        if not np.isfinite(yvals[i]):
            continue
        eg = elev_groups[i]
        rg = rot_groups[i]
        bg = band_groups[i] if band_groups is not None else None
        style = visit_marker_style(elev=eg, rot=rg, band=bg, base_size=7)
        ax.plot(visits_x[i], yvals[i], alpha=0.9, **style)
    ax.set_xlabel('visit index')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)
    _add_day_obs_dividers(ax, dobs_seq, label_top=True)
    return fig


def _sigma_pair(res):
    """(sigma_std, sigma_MAD) of finite values, or (NaN, NaN) if too few."""
    res = res[np.isfinite(res)]
    if len(res) < 5:
        return np.nan, np.nan
    sigma = float(np.std(res))
    mad = float(np.median(np.abs(res - np.median(res))))
    return sigma, 1.4826 * mad


# ---- Tracking pages (shape=rot, color=elev, edge=band, fill=iter) ----

def _plot_tracking_panel(ax, vidx_groups, v1, v2, e1, e2,
                         show_errorbars=True):
    """Plot iter-1 (filled) + iter-2 (open) grouped by (elev, rot, band)."""
    for (eg, rg, bg), vidx_arr in vidx_groups.items():
        style1 = visit_marker_style(elev=eg, rot=rg, band=bg,
                                    iter_=1, base_size=5)
        style2 = visit_marker_style(elev=eg, rot=rg, band=bg,
                                    iter_=2, base_size=5)
        v1_g = v1[vidx_arr]; v2_g = v2[vidx_arr]
        if show_errorbars:
            e1_g = e1[vidx_arr]; e2_g = e2[vidx_arr]
            ax.errorbar(vidx_arr, v1_g, yerr=e1_g,
                        elinewidth=0.5, capsize=0, alpha=0.85, **style1)
            ax.errorbar(vidx_arr, v2_g, yerr=e2_g,
                        elinewidth=0.5, capsize=0, alpha=0.85, **style2)
        else:
            ax.plot(vidx_arr, v1_g, alpha=0.85, **style1)
            ax.plot(vidx_arr, v2_g, alpha=0.85, **style2)


def per_term_pages(fit_rows_iter1, fit_rows_iter2, by_pupil,
                   coord_sys, visits_table=None,
                   dz_prefix='dz', plots_per_page=16, ncols=4,
                   skip_bad=True):
    """One panel per (k, j) -- fit coefficient vs visit, both iterations.

    Marker scheme: shape = rotator group, color = elevation group,
    edge color = filter band, fill = iteration. The first page is the
    shared marker-scheme legend.
    """
    pairs = sorted(((k, j) for j, ks in by_pupil.items() for k in ks),
                   key=lambda kj: (kj[0], kj[1]))
    idx1 = _good_indices(fit_rows_iter1) if skip_bad else list(
        range(len(fit_rows_iter1)))
    idx2 = _good_indices(fit_rows_iter2) if skip_bad else list(
        range(len(fit_rows_iter2)))

    visit_meta = (build_visit_marker_lookup(visits_table)
                  if visits_table is not None else {})
    dobs1, elev1, rot1, band1 = _per_visit_groups(
        fit_rows_iter1, idx1, visit_meta)

    # Pre-group visit indices by (elev, rot, band) so each panel's plot
    # calls are vectorized. iter-1 and iter-2 share indices because
    # idx1 / idx2 enumerate the same good visits in the same order.
    from collections import defaultdict
    vidx_groups = defaultdict(list)
    for i, (eg, rg, bg) in enumerate(zip(elev1, rot1, band1)):
        vidx_groups[(eg, rg, bg)].append(i)
    vidx_groups = {k: np.array(v) for k, v in vidx_groups.items()}

    n = len(pairs)
    nrows = plots_per_page // ncols
    pages = [markers_legend_figure(show_iter_distinction=True)]

    for page_start in range(0, n, plots_per_page):
        page = pairs[page_start:page_start + plots_per_page]
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(4.0 * ncols, 3.0 * nrows),
                                 layout='constrained')
        axes = np.atleast_1d(axes).ravel()
        for idx, (k, j) in enumerate(page):
            ax = axes[idx]
            v1 = _series(fit_rows_iter1, idx1, k, j, dz_prefix)
            v2 = _series(fit_rows_iter2, idx2, k, j, dz_prefix)
            e1 = _series(fit_rows_iter1, idx1, k, j, dz_prefix, err=True)
            e2 = _series(fit_rows_iter2, idx2, k, j, dz_prefix, err=True)
            _plot_tracking_panel(ax, vidx_groups, v1, v2, e1, e2,
                                 show_errorbars=True)
            ax.axhline(0, color='gray', lw=0.5, alpha=0.5)
            ax.set_title(f'(k={k}, j={j})', fontsize=10)
            ax.tick_params(labelsize=7)
            _add_day_obs_dividers(ax, dobs1, label_top=(idx == 0))
        for idx in range(len(page), len(axes)):
            axes[idx].set_visible(False)
        page_idx = page_start // plots_per_page + 1
        n_pages = int(np.ceil(n / plots_per_page))
        n_skipped = (len(fit_rows_iter1) - len(idx1)
                     if skip_bad else 0)
        suffix = (f'  (good visits only, dropped {n_skipped} bad-flagged)'
                  if skip_bad else '')
        fig.suptitle(
            f'DZ fit coefficients vs visit  '
            f'({page_idx}/{n_pages}, {coord_sys}){suffix}',
            fontsize=12)
        pages.append(fig)
    return pages


def per_term_robust_rms_pages(fit_rows_iter1, fit_rows_iter2, by_pupil,
                              coord_sys, visits_table=None,
                              dz_prefix='dz',
                              plots_per_page=16, ncols=4,
                              skip_bad=True):
    """One panel per (k, j) -- robust RMS (RLM bse) vs visit, both iters.

    Same marker scheme as per_term_pages.
    """
    pairs = sorted(((k, j) for j, ks in by_pupil.items() for k in ks),
                   key=lambda kj: (kj[0], kj[1]))
    idx1 = _good_indices(fit_rows_iter1) if skip_bad else list(
        range(len(fit_rows_iter1)))
    idx2 = _good_indices(fit_rows_iter2) if skip_bad else list(
        range(len(fit_rows_iter2)))

    visit_meta = (build_visit_marker_lookup(visits_table)
                  if visits_table is not None else {})
    dobs1, elev1, rot1, band1 = _per_visit_groups(
        fit_rows_iter1, idx1, visit_meta)

    from collections import defaultdict
    vidx_groups = defaultdict(list)
    for i, (eg, rg, bg) in enumerate(zip(elev1, rot1, band1)):
        vidx_groups[(eg, rg, bg)].append(i)
    vidx_groups = {k: np.array(v) for k, v in vidx_groups.items()}

    n = len(pairs)
    nrows = plots_per_page // ncols
    pages = [markers_legend_figure(show_iter_distinction=True)]

    for page_start in range(0, n, plots_per_page):
        page = pairs[page_start:page_start + plots_per_page]
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(4.0 * ncols, 3.0 * nrows),
                                 layout='constrained')
        axes = np.atleast_1d(axes).ravel()
        for idx, (k, j) in enumerate(page):
            ax = axes[idx]
            e1 = _series(fit_rows_iter1, idx1, k, j, dz_prefix, err=True)
            e2 = _series(fit_rows_iter2, idx2, k, j, dz_prefix, err=True)
            _plot_tracking_panel(ax, vidx_groups, e1, e2, e1, e2,
                                 show_errorbars=False)
            ax.set_title(f'(k={k}, j={j})  sigma_robust', fontsize=10)
            ax.set_ylim(bottom=0)
            ax.tick_params(labelsize=7)
            _add_day_obs_dividers(ax, dobs1, label_top=(idx == 0))
        for idx in range(len(page), len(axes)):
            axes[idx].set_visible(False)
        page_idx = page_start // plots_per_page + 1
        n_pages = int(np.ceil(n / plots_per_page))
        suffix = '  (good visits only)' if skip_bad else ''
        fig.suptitle(
            f'Per-coefficient robust RMS (RLM bse)  '
            f'({page_idx}/{n_pages}, {coord_sys}){suffix}',
            fontsize=12)
        pages.append(fig)
    return pages


tracking_figs = []
tracking_err_figs = []
if len(result['iter_results']) >= 2:
    rows1 = result['iter_results'][0]['fit_rows']
    rows2 = result['iter_results'][1]['fit_rows']
    n1_bad = sum(1 for r in rows1 if r.get('bad_fit', False))
    n2_bad = sum(1 for r in rows2 if r.get('bad_fit', False))
    print(f'Tracking plot: dropping {n1_bad} bad-flagged iter-1 visits '
          f'and {n2_bad} from iter-2.')

    tracking_figs = per_term_pages(rows1, rows2, by_pupil, coord_sys,
                                   visits_table=visits_kept, skip_bad=True)
    print(f'Built {len(tracking_figs)} per-DZ-term tracking pages '
          f'(includes legend page)')

    tracking_err_figs = per_term_robust_rms_pages(
        rows1, rows2, by_pupil, coord_sys,
        visits_table=visits_kept, skip_bad=True)
    print(f'Built {len(tracking_err_figs)} per-DZ-term robust-RMS pages '
          f'(includes legend page)')
else:
    print('Skipping tracking pages: only one iteration available')

<a id='validation'></a>
## 11. Per-visit Residual Validation

For each (good) visit, compute the per-donut residual

   residual = zk_data − (measured_intrinsic_iter2 + DZ_fit_iter2)

evaluated at each donut's (thx, thy).  The Z4 corrections from §5 are
already baked into `zk_data` and the iter-2 grid, so the residual is
height-aware.

Pages are streamed directly to `measured_intrinsic_validation.pdf`.

Per-visit pages are *subsampled* via `hist_visit_stride` (default 12 —
one page per FAM triplet group): histograms (and optional maps) are
only emitted every `hist_visit_stride` visits.  σ / σ_MAD summary
stats are still collected for **every** good visit, so the time-series
plots cover the whole run.

Page order:

1. Per-visit histograms (subsampled) — 4×4 grid of 1-D residual
   histograms for pupil j = 4..19.
2. Per-visit maps (optional, also subsampled) — `make_residual_maps`
   toggle.
3. Marker legend page.
4. **Quadrature** σ vs visit  —  σ_quad = √(Σ_j σ_j²) over `residual_j_range`.
5. **Quadrature** σ_MAD vs visit  —  same with σ_MAD,j.
6. Per-j σ vs visit and σ_MAD vs visit, for j in `per_zernike_sigma_j`
   (default 4..8).

Marker shape encodes rotator group; marker color encodes elevation
group.  The §10 tracking PDFs use the same scheme, with **iter-1 = filled
marker** and **iter-2 = open marker**.

In [ ]:
def compute_validation_residual(donut_df, result, coord_sys, iZs):
    """Per-donut residual after subtracting iter-2 intrinsic AND iter-2 DZ fit."""
    final_iter = result['iter_results'][-1]
    wfd_sub = final_iter['wfd_subtracted']
    measured_grid = final_iter['measured_grid']
    xcent, ycent = result['xcent'], result['ycent']
    thx = np.rad2deg(np.asarray(donut_df[f'thx_{coord_sys}'], dtype=float))
    thy = np.rad2deg(np.asarray(donut_df[f'thy_{coord_sys}'], dtype=float))
    intrinsic_at_donut = interpolate_grid_at_donuts(
        measured_grid, xcent, ycent, thx, thy, iZs)
    return wfd_sub - intrinsic_at_donut


def per_visit_validation_to_pdf(donut_df, result, coord_sys, iZs,
                                output_pdf_path,
                                j_range, hist_range_um=(-1.0, 1.0),
                                n_hist_bins=60,
                                map_n_bins=16, map_fp_radius=1.8,
                                make_maps=False, ncols=4,
                                visits_table=None,
                                per_j_sigma_range=None,
                                show_first=True,
                                hist_visit_stride=1):
    """Stream per-visit residual histograms (and optional maps) plus a
    quadrature-summed σ summary block directly to ``output_pdf_path``.

    The summary block emits, in this order:
      * Marker legend page (no iter distinction — only iter-2 is used).
      * Quadrature σ vs visit: σ_total[v] = sqrt(Σ_j σ_j(v)^2) over j_list.
      * Quadrature σ_MAD vs visit:
            σ_MAD_total[v] = sqrt(Σ_j σ_MAD_j(v)^2).
      * For each j in `per_j_sigma_range`: σ_j vs visit.
      * For each j in `per_j_sigma_range`: σ_MAD_j vs visit.
    """
    from scipy.stats import binned_statistic_2d

    residuals = compute_validation_residual(donut_df, result, coord_sys, iZs)
    iZidx = {iZ: i for i, iZ in enumerate(iZs)}
    j_list = [j for j in j_range if j in iZidx]
    n = len(j_list)
    nrows = (n + ncols - 1) // ncols

    final_iter = result['iter_results'][-1]
    good_rows = [r for r in final_iter['fit_rows']
                 if not r.get('bad_fit', False)]

    dobs_full = np.asarray(donut_df['day_obs'])
    snum_full = np.asarray(donut_df['seq_num'])

    j_per = [j for j in (per_j_sigma_range
                         if per_j_sigma_range is not None
                         else range(4, 9))
             if j in iZidx]

    visit_meta = (_build_visit_lookup(visits_table)
                  if visits_table is not None else {})
    # σ_j and σ_MAD_j for *every* j in j_list (needed for quadrature),
    # plus an explicit per_j subset for the per-Zernike pages.
    sigma_per_j_all     = {j: [] for j in j_list}
    sigma_mad_per_j_all = {j: [] for j in j_list}
    dobs_seq, elev_groups, rot_groups = [], [], []

    Path(output_pdf_path).parent.mkdir(parents=True, exist_ok=True)
    n_pages = 0
    n_pages_per_emitting_visit = 2 if make_maps else 1
    n_emitting_visits = (len(good_rows)
                         + hist_visit_stride - 1) // hist_visit_stride
    print(f'Streaming validation PDF -> {output_pdf_path}')
    print(f'  {len(good_rows)} good visits; emitting histogram'
          f'{"+map" if make_maps else ""} pages every '
          f'{hist_visit_stride} visit(s) -> {n_emitting_visits} '
          f'visit-page(s) × {n_pages_per_emitting_visit}')
    print(f'  σ / σ_MAD summaries still cover ALL good visits  '
          f'(j = {j_list[0]}..{j_list[-1]}, hist range {hist_range_um})')

    with PdfPages(output_pdf_path) as pdf:
        for r_idx, r in enumerate(good_rows):
            d = int(r['day_obs'])
            s = int(r['seq_num'])
            mask = (dobs_full == d) & (snum_full == s)
            if not np.any(mask):
                continue
            n_donuts = int(mask.sum())

            if (r_idx % hist_visit_stride) == 0:
                # ---- Histogram page ----
                fig, axes = plt.subplots(nrows, ncols,
                                         figsize=(3.5 * ncols, 2.8 * nrows),
                                         layout='constrained')
                axes = np.atleast_1d(axes).ravel()
                for idx, j in enumerate(j_list):
                    ax = axes[idx]
                    res = residuals[mask, iZidx[j]]
                    res = res[np.isfinite(res)]
                    if len(res) < 2:
                        ax.set_visible(False)
                        continue
                    ax.hist(res, bins=n_hist_bins, range=hist_range_um,
                            edgecolor='black', linewidth=0.3, alpha=0.75)
                    sigma = float(np.std(res))
                    mad = float(np.median(np.abs(res - np.median(res))))
                    sigma_mad_val = 1.4826 * mad
                    n_in = int(np.sum((res >= hist_range_um[0])
                                      & (res <= hist_range_um[1])))
                    ax.text(0.97, 0.95,
                            f'σ={sigma:.3f}\nσ_MAD={sigma_mad_val:.3f}\n'
                            f'{n_in}/{len(res)} in range',
                            transform=ax.transAxes, ha='right', va='top',
                            fontsize=8,
                            bbox=dict(boxstyle='round', facecolor='wheat',
                                      alpha=0.5))
                    ax.axvline(0, color='gray', lw=0.5, alpha=0.5)
                    ax.set_title(f'j={j}', fontsize=10)
                    ax.set_xlabel('residual [μm]', fontsize=8)
                    ax.tick_params(labelsize=7)
                    ax.set_xlim(*hist_range_um)
                for idx in range(len(j_list), len(axes)):
                    axes[idx].set_visible(False)
                fig.suptitle(
                    f'Visit ({d}, {s})  [{r_idx + 1}/{len(good_rows)}]  — '
                    f'residual histograms (data − iter2_intrinsic − iter2_DZ, '
                    f'n_donuts={n_donuts})',
                    fontsize=12)
                if show_first and r_idx == 0:
                    plt.show()
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
                n_pages += 1

                # ---- Optional residual map page ----
                if make_maps:
                    xbins_m = np.linspace(-map_fp_radius, map_fp_radius,
                                          map_n_bins + 1)
                    ybins_m = np.linspace(-map_fp_radius, map_fp_radius,
                                          map_n_bins + 1)
                    thx_v = np.rad2deg(np.asarray(donut_df[f'thx_{coord_sys}'],
                                                  dtype=float))[mask]
                    thy_v = np.rad2deg(np.asarray(donut_df[f'thy_{coord_sys}'],
                                                  dtype=float))[mask]

                    fig, axes = plt.subplots(nrows, ncols,
                                             figsize=(3.5 * ncols, 3.2 * nrows),
                                             layout='constrained')
                    axes = np.atleast_1d(axes).ravel()
                    for idx, j in enumerate(j_list):
                        ax = axes[idx]
                        res = residuals[mask, iZidx[j]]
                        if not np.any(np.isfinite(res)):
                            ax.set_visible(False)
                            continue
                        stat_val, _, _, _ = binned_statistic_2d(
                            thy_v, thx_v, res, statistic='median',
                            bins=[xbins_m, ybins_m])
                        vmin, vmax = np.nanpercentile(res, [5, 95])
                        im = ax.imshow(
                            stat_val.T, origin='lower',
                            extent=[xbins_m[0], xbins_m[-1],
                                    ybins_m[0], ybins_m[-1]],
                            cmap='RdBu_r', interpolation='none',
                            aspect='equal',
                            vmin=vmin, vmax=vmax)
                        plt.colorbar(im, ax=ax, shrink=0.65, label='μm')
                        ax.set_title(f'j={j}', fontsize=10)
                        ax.set_xlabel(f'thy_{coord_sys} [deg]', fontsize=8)
                        ax.set_ylabel(f'thx_{coord_sys} [deg]', fontsize=8)
                        ax.tick_params(labelsize=7)
                    for idx in range(len(j_list), len(axes)):
                        axes[idx].set_visible(False)
                    fig.suptitle(
                        f'Visit ({d}, {s})  [{r_idx + 1}/{len(good_rows)}]  — '
                        f'residual maps ({map_n_bins}×{map_n_bins} bins, '
                        f'n_donuts={n_donuts})',
                        fontsize=12)
                    pdf.savefig(fig, bbox_inches='tight')
                    plt.close(fig)
                    n_pages += 1

            # ---- Per-visit summary collection (per j across full j_list) ----
            any_finite = False
            visit_per_j     = {}
            visit_per_j_mad = {}
            for j in j_list:
                res_j = residuals[mask, iZidx[j]]
                sj, smj = _sigma_pair(res_j)
                visit_per_j[j] = sj
                visit_per_j_mad[j] = smj
                if np.isfinite(sj):
                    any_finite = True
            if not any_finite:
                continue
            for j in j_list:
                sigma_per_j_all[j].append(visit_per_j[j])
                sigma_mad_per_j_all[j].append(visit_per_j_mad[j])
            dobs_seq.append(d)
            meta = visit_meta.get((d, s),
                                  {'alt_deg': np.nan, 'rot_deg': np.nan})
            elev_groups.append(_elev_group(meta['alt_deg']))
            rot_groups.append(_rot_group(meta['rot_deg']))

            if (r_idx + 1) % 10 == 0:
                print(f'    ...wrote {r_idx + 1}/{len(good_rows)} visits')

        # Drop the heavy residuals array now that all per-visit work is done.
        del residuals

        # ---- Summary block ----
        if visits_table is not None and len(dobs_seq) > 0:
            n_v = len(dobs_seq)
            visits_x = np.arange(n_v)

            # Quadrature σ_total[v] = sqrt(Σ_j σ_j(v)^2) over j_list
            sigma_quad = np.zeros(n_v)
            sigma_mad_quad = np.zeros(n_v)
            for v in range(n_v):
                acc_s = 0.0; acc_m = 0.0
                n_used_s = 0; n_used_m = 0
                for j in j_list:
                    sj = sigma_per_j_all[j][v]
                    smj = sigma_mad_per_j_all[j][v]
                    if np.isfinite(sj):
                        acc_s += sj * sj
                        n_used_s += 1
                    if np.isfinite(smj):
                        acc_m += smj * smj
                        n_used_m += 1
                sigma_quad[v] = np.sqrt(acc_s) if n_used_s > 0 else np.nan
                sigma_mad_quad[v] = (np.sqrt(acc_m)
                                     if n_used_m > 0 else np.nan)

            pool_lo, pool_hi = j_list[0], j_list[-1]

            for fig in [_markers_legend_page(show_iter_distinction=False)]:
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
                n_pages += 1

            for ylabel, yvals, title in [
                ('σ_quad = √(Σ σ_j²) [μm]', sigma_quad,
                 f'Quadrature-summed per-visit residual σ vs visit  '
                 f'({coord_sys}; j={pool_lo}..{pool_hi}, '
                 f'n_visits={n_v})'),
                ('σ_MAD,quad = √(Σ σ_MAD,j²) [μm]', sigma_mad_quad,
                 f'Quadrature-summed per-visit residual σ_MAD vs visit  '
                 f'({coord_sys}; j={pool_lo}..{pool_hi})'),
            ]:
                fig = _make_sigma_time_plot(
                    visits_x, yvals, dobs_seq, elev_groups, rot_groups,
                    ylabel=ylabel, title=title)
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
                n_pages += 1

            for j in j_per:
                fig = _make_sigma_time_plot(
                    visits_x, np.array(sigma_per_j_all[j]), dobs_seq,
                    elev_groups, rot_groups,
                    ylabel='σ (std) [μm]',
                    title=f'Per-visit residual σ vs visit  '
                          f'({coord_sys}; j={j})')
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
                n_pages += 1

            for j in j_per:
                fig = _make_sigma_time_plot(
                    visits_x, np.array(sigma_mad_per_j_all[j]), dobs_seq,
                    elev_groups, rot_groups,
                    ylabel='σ_MAD = 1.4826·MAD [μm]',
                    title=f'Per-visit residual σ_MAD vs visit  '
                          f'({coord_sys}; j={j})')
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
                n_pages += 1

    print(f'Wrote {n_pages} pages.')
    return n_pages


validation_pdf_path = None
if len(result['iter_results']) >= 2:
    if output_dir is None:
        output_dir = output_dir_default(
            binning, n_keep_eff, coord_sys,
            day_obs_min, day_obs_max,
            seq_num_min, seq_num_max,
            alt_min_deg, alt_max_deg,
            rot_min_deg, rot_max_deg)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    validation_pdf_path = str(output_dir / 'measured_intrinsic_validation.pdf')

    n_val_pages = per_visit_validation_to_pdf(
        donut_df, result, coord_sys, iZs,
        output_pdf_path=validation_pdf_path,
        j_range=residual_j_range,
        hist_range_um=residual_hist_range,
        n_hist_bins=residual_n_hist_bins,
        map_n_bins=residual_map_n_bins,
        map_fp_radius=residual_map_fp_radius,
        make_maps=make_residual_maps,
        visits_table=visits_kept,
        per_j_sigma_range=per_zernike_sigma_j,
        show_first=True,
        hist_visit_stride=hist_visit_stride)
else:
    print('Skipping validation pages: only one iteration available')

<a id='output'></a>
## 12. Output Tables

In [ ]:
if output_dir is None:
    output_dir = output_dir_default(
        binning, n_keep_eff, coord_sys,
        day_obs_min, day_obs_max,
        seq_num_min, seq_num_max,
        alt_min_deg, alt_max_deg,
        rot_min_deg, rot_max_deg)
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {output_dir}')
print(f'Output label:     {output_label}')


def _save_path_outputs(result_obj, tag, intrinsic_grid_for_table):
    """Write per-path DZ fit parquet + measured-intrinsic grid table."""
    final_iter = result_obj['iter_results'][-1]
    dz_fits_path = output_dir / f'measured_intrinsic_{output_label}_{tag}_dz_fits.parquet'
    save_dz_fits(final_iter['fit_rows'], dz_fits_path)

    xc, yc = result_obj['xcent'], result_obj['ycent']
    tbl = assemble_intrinsic_table(
        grid=intrinsic_grid_for_table, iZs=iZs, xcent=xc, ycent=yc,
        coord_sys_grid=coord_sys, alt_coord_xform=None)
    print(f'  [{tag}] grid table: {len(tbl)} non-empty bins in {coord_sys}')

    out_path = output_dir / f'measured_intrinsic_{output_label}_{tag}_grid.{output_format}'
    if output_format == 'parquet':
        tbl.write(str(out_path), format='parquet', overwrite=True)
    elif output_format == 'fits':
        tbl.write(str(out_path), format='fits', overwrite=True)
    elif output_format == 'hdf5':
        tbl.write(str(out_path), format='hdf5', path='grid',
                  overwrite=True, append=False)
    else:
        raise ValueError(f'Unknown output_format: {output_format}')
    print(f'  [{tag}] saved measured intrinsic grid: {out_path}')


# Path B: build a per-pupil-j stack from the iter-final grid
_save_path_outputs(result_B, 'pathB', iter2_grids)

# Path A: same, from result_A
iter2_grids_A = result_A['iter_results'][-1]['measured_grid']
_save_path_outputs(result_A, 'pathA', iter2_grids_A)


def _write_pdf(figs, fname):
    p = output_dir / fname
    with PdfPages(p) as pdf:
        for fig in figs:
            pdf.savefig(fig, bbox_inches='tight')
    print(f'  wrote {p}')


# --- PDFs ----------------------------------------------------------------
print('\nWriting PDFs:')
_write_pdf(comparison_figs,
           f'measured_intrinsic_{output_label}_pathB_comparison.pdf')

if 'compare_figs' in globals() and compare_figs:
    _write_pdf(compare_figs,
               f'measured_intrinsic_{output_label}_pathA_vs_pathB.pdf')

if 'validation_A_figs' in globals() and validation_A_figs:
    _write_pdf(validation_A_figs,
               f'measured_intrinsic_{output_label}_pathA_validation.pdf')

if tracking_figs:
    _write_pdf(tracking_figs,
               f'measured_intrinsic_{output_label}_pathB_tracking.pdf')

if tracking_err_figs:
    _write_pdf(tracking_err_figs,
               f'measured_intrinsic_{output_label}_pathB_tracking_robust_rms.pdf')

if validation_pdf_path is not None:
    print(f'\nValidation PDF (streamed in §15): {validation_pdf_path}')


<a id='format'></a>
## 13. Output Format Options — Rubin DM datasets

The "measured intrinsic" map is a **per-pupil-j focal-plane grid of
median Zernike values** — conceptually parallel to the existing batoid
intrinsic table (`zk_intrinsic_*` columns produced upstream by
`aggregateAOSVisitTable`), but data-derived.  Realistic choices:

| Option | Pros | Cons |
|---|---|---|
| **Native parquet** (default) | Matches every other AOS pipeline output; loadable by `astropy.QTable.read` and `pandas.read_parquet`; round-trips list columns natively. | Not registered with the Butler — passed by path. |
| **FITS BinTable** | DM-friendly, viewable in DS9/topcat. | Slightly heavier; list columns become variable-length arrays. |
| **HDF5 QTable** | Multi-table file possible. | Less standard for AOS outputs going forward. |
| **Custom Butler dataset type** | Full provenance, `butler.get(...)`. | Requires an RFC + StorageClass + dimension records. |

Iterating with parquet for now; once the iteration logic is stable a
Butler dataset type is a natural next step.